# Tutorial 08: De Bruijn Graphs — Sequence Reconstruction

This tutorial introduces the `debruijn` module, which provides a generic De Bruijn graph implementation for reconstructing sequences from overlapping fragments.

## What You'll Learn

1. **De Bruijn Graph Concept** — Nodes, edges, and k-mers
2. **Building Graphs** — Adding k-mers and multiplicities
3. **Eulerian Paths** — Visiting each edge exactly once
4. **Sequence Reconstruction** — Converting paths to sequences
5. **Boundary Markers** — START/END tokens for anchoring

## The De Bruijn Graph Structure

For k-mers of size k:
- **Nodes** = (k-1)-mers (prefixes and suffixes)
- **Edges** = k-mers, connecting prefix to suffix
- **Edge multiplicity** = how many times the k-mer appears

For trigrams (k=3):
```
Trigram: (A, B, C)
  - Node (A, B) → Node (B, C)
  - Edge label: C
```

**Eulerian Path**: A path that visits each edge exactly once (per multiplicity).
This is the key to reconstructing the original sequence from overlapping k-mers.

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.debruijn import (
    DeBruijnGraph,
    Edge,
    PathResult,
)

## 1. Creating a De Bruijn Graph

Let's start with a simple example: reconstructing "A B C D" from trigrams.

In [2]:
# Create a De Bruijn graph for trigrams (k=3)
graph = DeBruijnGraph(k=3)

print(f"De Bruijn graph: k={graph.k}")
print(f"  Nodes: {graph.num_nodes}")
print(f"  Edges: {graph.num_edges}")

De Bruijn graph: k=3
  Nodes: 0
  Edges: 0


In [3]:
# Add trigrams from sequence "START A B C D END"
# For sequence reconstruction, we add START/END markers

trigrams = [
    ("START", "A", "B"),
    ("A", "B", "C"),
    ("B", "C", "D"),
    ("C", "D", "END"),
]

for trigram in trigrams:
    edge = graph.add_kmer(trigram)
    print(f"Added: {trigram}")
    print(f"  Edge: {edge.source} → {edge.target}, label='{edge.label}'")

Added: ('START', 'A', 'B')
  Edge: ('START', 'A') → ('A', 'B'), label='B'
Added: ('A', 'B', 'C')
  Edge: ('A', 'B') → ('B', 'C'), label='C'
Added: ('B', 'C', 'D')
  Edge: ('B', 'C') → ('C', 'D'), label='D'
Added: ('C', 'D', 'END')
  Edge: ('C', 'D') → ('D', 'END'), label='END'


In [4]:
# Check graph structure
print(f"\nGraph structure:")
print(f"  Nodes: {graph.num_nodes}")
print(f"  Edges: {graph.num_edges}")
print(f"  Total edge count (with multiplicity): {graph.total_edge_count}")
print(f"\nNodes: {graph.nodes}")


Graph structure:
  Nodes: 5
  Edges: 4
  Total edge count (with multiplicity): 4

Nodes: {('B', 'C'), ('D', 'END'), ('START', 'A'), ('A', 'B'), ('C', 'D')}


## 2. Finding Eulerian Paths

An Eulerian path visits each edge exactly once. For sequence reconstruction, this path gives us the original token order.

In [5]:
# Find Eulerian path
result = graph.find_eulerian_path(start_marker="START", end_marker="END")

print(f"Eulerian path found: {result.is_eulerian}")
print(f"\nPath (nodes visited):")
for i, node in enumerate(result.path):
    print(f"  [{i}] {node}")

Eulerian path found: True

Path (nodes visited):
  [0] ('START', 'A')
  [1] ('A', 'B')
  [2] ('B', 'C')
  [3] ('C', 'D')
  [4] ('D', 'END')


In [6]:
# Get reconstructed sequence
print(f"\nReconstructed sequence:")
print(f"  {result.sequence}")
print(f"\nOriginal: ['START', 'A', 'B', 'C', 'D', 'END']")


Reconstructed sequence:
  ['START', 'A', 'B', 'C', 'D', 'END']

Original: ['START', 'A', 'B', 'C', 'D', 'END']


## 3. Handling Repeated Patterns

When a k-mer appears multiple times, we track its **multiplicity**. The Eulerian path uses each edge exactly as many times as its multiplicity.

In [7]:
# Sequence with repetition: "START A B A B A END"
# Note: (A, B, A) appears twice!

graph_repeat = DeBruijnGraph(k=3)

trigrams_repeat = [
    ("START", "A", "B"),
    ("A", "B", "A"),  # First occurrence
    ("B", "A", "B"),
    ("A", "B", "A"),  # Second occurrence (duplicate!)
    ("B", "A", "END"),
]

for trigram in trigrams_repeat:
    graph_repeat.add_kmer(trigram)

# Check multiplicity of repeated trigram
edge = graph_repeat.get_edge(("A", "B", "A"))
print(f"Edge (A, B, A): multiplicity = {edge.multiplicity}")

print(f"\nGraph: {graph_repeat.num_nodes} nodes, {graph_repeat.num_edges} unique edges")
print(f"Total edge count (with multiplicity): {graph_repeat.total_edge_count}")

Edge (A, B, A): multiplicity = 2

Graph: 4 nodes, 4 unique edges
Total edge count (with multiplicity): 5


In [8]:
# Find Eulerian path - handles multiplicities correctly
result_repeat = graph_repeat.find_eulerian_path(start_marker="START", end_marker="END")

print(f"Eulerian path found: {result_repeat.is_eulerian}")
print(f"\nReconstructed sequence:")
print(f"  {result_repeat.sequence}")
print(f"\nOriginal: ['START', 'A', 'B', 'A', 'B', 'A', 'END']")

Eulerian path found: True

Reconstructed sequence:
  ['START', 'A', 'B', 'A', 'B', 'A', 'END']

Original: ['START', 'A', 'B', 'A', 'B', 'A', 'END']


## 4. Degree Analysis

For an Eulerian path to exist:
- At most one node has `out_degree - in_degree = +1` (start)
- At most one node has `out_degree - in_degree = -1` (end)
- All other nodes have `out_degree = in_degree`

In [9]:
# Analyze node degrees
print(f"Node degree analysis:")
for node in sorted(graph.nodes):
    in_deg = graph.in_degree(node)
    out_deg = graph.out_degree(node)
    balance = graph.degree_balance(node)
    status = "(start)" if balance == 1 else "(end)" if balance == -1 else ""
    print(f"  {node}: in={in_deg}, out={out_deg}, balance={balance:+d} {status}")

Node degree analysis:
  ('A', 'B'): in=1, out=1, balance=+0 
  ('B', 'C'): in=1, out=1, balance=+0 
  ('C', 'D'): in=1, out=1, balance=+0 
  ('D', 'END'): in=1, out=0, balance=-1 (end)
  ('START', 'A'): in=0, out=1, balance=+1 (start)


In [10]:
# Find start/end nodes automatically
start_node, end_node = graph.find_start_end_nodes(start_marker="START", end_marker="END")

print(f"Start node: {start_node}")
print(f"End node: {end_node}")

Start node: ('START', 'A')
End node: ('D', 'END')


## 5. Edge Inspection

You can inspect outgoing and incoming edges for each node.

In [11]:
# Inspect edges
print(f"Edge structure:")
for node in sorted(graph.nodes):
    out_edges = graph.out_edges(node)
    if out_edges:
        print(f"\n  From {node}:")
        for edge in out_edges:
            print(f"    → {edge.target} (label='{edge.label}', mult={edge.multiplicity})")

Edge structure:

  From ('A', 'B'):
    → ('B', 'C') (label='C', mult=1)

  From ('B', 'C'):
    → ('C', 'D') (label='D', mult=1)

  From ('C', 'D'):
    → ('D', 'END') (label='END', mult=1)

  From ('START', 'A'):
    → ('A', 'B') (label='B', mult=1)


## 6. Practical Example: Text Reconstruction

Let's reconstruct a sentence from its trigrams.

In [12]:
# Original sentence
sentence = "the quick brown fox jumps over the lazy dog"
tokens = ["<S>"] + sentence.split() + ["</S>"]

print(f"Original tokens: {tokens}")

Original tokens: ['<S>', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '</S>']


In [13]:
# Build De Bruijn graph from trigrams
text_graph = DeBruijnGraph(k=3)

for i in range(len(tokens) - 2):
    trigram = tuple(tokens[i:i+3])
    text_graph.add_kmer(trigram)

print(f"Graph built:")
print(f"  Nodes (bigrams): {text_graph.num_nodes}")
print(f"  Edges (trigrams): {text_graph.num_edges}")

Graph built:
  Nodes (bigrams): 10
  Edges (trigrams): 9


In [14]:
# Reconstruct
result = text_graph.find_eulerian_path(start_marker="<S>", end_marker="</S>")

print(f"Reconstruction successful: {result.is_eulerian}")
print(f"\nReconstructed: {' '.join(result.sequence)}")
print(f"Original:      {' '.join(tokens)}")
print(f"\nMatch: {result.sequence == tokens}")

Reconstruction successful: True

Reconstructed: <S> the quick brown fox jumps over the lazy dog </S>
Original:      <S> the quick brown fox jumps over the lazy dog </S>

Match: True


## 7. Handling Ambiguous Cases

When the graph has multiple valid Eulerian paths, randomization can explore alternatives.

In [15]:
# Ambiguous case: "A B C" and "A B D" share prefix "A B"
# Combined trigrams could produce either sequence

ambig_graph = DeBruijnGraph(k=3)

# Two possible sequences merged
ambig_graph.add_kmer(("<S>", "A", "B"))
ambig_graph.add_kmer(("A", "B", "C"))
ambig_graph.add_kmer(("B", "C", "</S>"))

# Find path (deterministic)
result1 = ambig_graph.find_eulerian_path(start_marker="<S>", end_marker="</S>")
print(f"Deterministic path: {result1.sequence}")

# Find path (randomized) - may give same result for simple cases
result2 = ambig_graph.find_eulerian_path(start_marker="<S>", end_marker="</S>", randomize=True)
print(f"Randomized path:    {result2.sequence}")

Deterministic path: ['<S>', 'A', 'B', 'C', '</S>']
Randomized path:    ['<S>', 'A', 'B', 'C', '</S>']


## 8. Batch K-mer Addition

For efficiency, add multiple k-mers at once with optional counts.

In [16]:
# Batch addition
batch_graph = DeBruijnGraph(k=3)

kmers = [
    ("<S>", "hello", "world"),
    ("hello", "world", "</S>"),
]

batch_graph.add_kmers(kmers)

result = batch_graph.find_eulerian_path(start_marker="<S>")
print(f"Batch result: {result.sequence}")

Batch result: ['<S>', 'hello', 'world', '</S>']


In [17]:
# Batch with counts
counted_graph = DeBruijnGraph(k=3)

kmers = [
    ("<S>", "a", "b"),
    ("a", "b", "a"),
    ("b", "a", "b"),
    ("a", "b", "</S>"),
]

counts = {
    ("a", "b", "a"): 2,  # Appears twice in sequence
    ("b", "a", "b"): 1,
}

counted_graph.add_kmers(kmers, counts=counts)

print(f"Edge (a,b,a) multiplicity: {counted_graph.get_edge(('a','b','a')).multiplicity}")

Edge (a,b,a) multiplicity: 2


## 9. Greedy Path Finding

When a full Eulerian path doesn't exist, greedy traversal gives a partial reconstruction.

In [18]:
# Incomplete graph (missing edges)
partial_graph = DeBruijnGraph(k=3)

partial_graph.add_kmer(("<S>", "A", "B"))
partial_graph.add_kmer(("A", "B", "C"))
# Missing: ("B", "C", "D") - creates a gap
partial_graph.add_kmer(("C", "D", "</S>"))

# Eulerian path fails
eul_result = partial_graph.find_eulerian_path(start_marker="<S>")
print(f"Eulerian path complete: {eul_result.is_eulerian if eul_result else 'No path'}")

# Greedy gives partial
greedy_result = partial_graph.find_path_greedy(start_marker="<S>")
if greedy_result:
    print(f"Greedy path: {greedy_result.sequence}")

Eulerian path complete: False
Greedy path: ['<S>', 'A', 'B', 'C']


## 10. PathResult Details

The `PathResult` contains full information about the reconstruction.

In [19]:
# Build a complete example
demo_graph = DeBruijnGraph(k=3)
demo_tokens = ["<S>", "one", "two", "three", "</S>"]

for i in range(len(demo_tokens) - 2):
    demo_graph.add_kmer(tuple(demo_tokens[i:i+3]))

result = demo_graph.find_eulerian_path(start_marker="<S>")

print(f"PathResult details:")
print(f"  is_eulerian: {result.is_eulerian}")
print(f"  sequence: {result.sequence}")
print(f"  path length: {len(result.path)} nodes")
print(f"  edges used: {len(result.edges_used)}")
print(f"  unused edges: {result.unused_edges}")

PathResult details:
  is_eulerian: True
  sequence: ['<S>', 'one', 'two', 'three', '</S>']
  path length: 4 nodes
  edges used: 3
  unused edges: {}


## Summary

In this tutorial, you learned:

1. **De Bruijn Structure** — Nodes are (k-1)-mers, edges are k-mers
2. **Edge Multiplicities** — Track repeated k-mers
3. **Eulerian Paths** — Visit each edge exactly once per multiplicity
4. **Sequence Reconstruction** — Convert paths back to token sequences
5. **Boundary Markers** — START/END tokens anchor the path

### Key Algorithms

| Algorithm | Use Case | Completeness |
|-----------|----------|-------------|
| `find_eulerian_path()` | Full reconstruction | All edges used exactly once |
| `find_path_greedy()` | Partial reconstruction | Stops at dead ends |

### Connection to HLLSet Framework

De Bruijn graphs are the foundation for **token order restoration**:
- **Tutorial 09**: HLLSet De Bruijn — D/R/N transformations as edges
- **Tutorial 10**: Disambiguation — Recover tokens from HLLSet fingerprints

### Next Steps

- **Tutorial 09**: HLLSet De Bruijn — Apply De Bruijn to HLLSet evolution